# Importing Libraries

In [2]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder

import pickle



# Data Ingestion

In [3]:
df = pd.read_csv(r"D:\PROJECTS\ANN_project_deployment\data\Churn_Modelling.csv")
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


# Understanding the data


## Data Cleaning

In [4]:
df = df.drop(["RowNumber","CustomerId","Surname"], axis= 1,)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [5]:
df.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

## Numerical vs Categorical Data

In [6]:
df["Geography"].unique()

<ArrowStringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [7]:
one_hot = OneHotEncoder()
geography_encoded = one_hot.fit_transform(df[["Geography"]])

geography_encoded

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [8]:
one_hot.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
geography_encoded_array = geography_encoded.toarray()
geography_encoded_dataframe = pd.DataFrame(geography_encoded_array,
                                            columns=one_hot.get_feature_names_out(["Geography"]))
geography_encoded_dataframe

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
df = pd.concat([df.drop("Geography",axis=1),geography_encoded_dataframe],axis =1)
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,Female,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Female,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,Female,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,Female,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Female,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,Male,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,Male,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,Female,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,Male,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [11]:
label_encode = LabelEncoder()
df["Gender"]=label_encode.fit_transform(df["Gender"])
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


## save the model for later usage

In [12]:
with open("one_hot_encoder_geography.pkl",'wb') as file:
    pickle.dump(one_hot,file)

with open("label_encoder_gender.pkl",'wb') as file:
    pickle.dump(label_encode,file)

## Independent and Dependendet features extraction

In [13]:
df.shape

(10000, 13)

In [14]:
X = df.drop("Exited",axis =1)
y = df["Exited"]


### train and test split

In [15]:
X_train,X_test,y_train,y_test = train_test_split(
                                        X,y,
                                        random_state=42,
                                        test_size= 0.2
)

### scaling

In [16]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [17]:
with open("standard_scaler.pkl","wb") as file:
    pickle.dump(scaler,file)

# ANN implementation

In [18]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

import datetime

## Build Our ANN model

In [19]:
X_train.shape[1] 
#number of nodes needs to input layer

12

In [20]:
num_of_input_features = X_train.shape[1]

## creating the sequential networks

In [21]:

model = Sequential([
    Input(shape=(num_of_input_features,)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

## compile the model

In [23]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

In [24]:
model.compile(optimizer= optimizer, #optimizer = "adam" can be used but it has a fixed learning rate so custom it
              loss = "binary_crossentropy",
              metrics = ["accuracy"])

In [25]:
import tensorflow as tf

print(tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

2.21.0
GPUs: []


In [26]:
tf.config.list_physical_devices('GPU')

[]

pip install tensorflow-directml-plugin

Microsoft provides DirectML support for AMD and NVIDIA GPUs.

# Setup TENSORBOARD


In [27]:
log_dir = "logs/fit/"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)


## Setup Early Stopping for avoiding Overfitting

In [28]:
early_stopping_callback = EarlyStopping(
                            monitor="val_loss", # monitor on loss parameter throughout the epochs
                            patience=10, # atleast wait for 10 epochs before stopping
                            verbose= 1,
                            restore_best_weights=True # restore and load the best weight 
)

# Train the model

In [29]:
history = model.fit(
    X_train, y_train,
    validation_data = (X_test,y_test),
    epochs = 100,
    callbacks = [tensorboard_callback,early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8321 - loss: 0.4000 - val_accuracy: 0.8570 - val_loss: 0.3730
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8545 - loss: 0.3560 - val_accuracy: 0.8580 - val_loss: 0.3525
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8586 - loss: 0.3461 - val_accuracy: 0.8565 - val_loss: 0.3507
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8610 - loss: 0.3415 - val_accuracy: 0.8600 - val_loss: 0.3495
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8583 - loss: 0.3403 - val_accuracy: 0.8610 - val_loss: 0.3402
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8583 - loss: 0.3412 - val_accuracy: 0.8570 - val_loss: 0.3472
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8618 - loss: 0.3351 - val_accuracy: 0.8590 - val_loss: 0.3465
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8624 - loss: 0.3333 - val_accu

# save the model in h5 file..compatible with keras

In [30]:
model.save("model.h5")

In [31]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


# load tensorboard extension

In [32]:
import sys
print(sys.executable)

d:\PROJECTS\ANN_project_deployment\venv\python.exe


In [33]:
# ERROR: Failed to launch TensorBoard (exited with 1).
# Contents of stderr:
# Traceback (most recent call last):
#   File "<frozen runpy>", line 198, in _run_module_as_main
#   File "<frozen runpy>", line 88, in _run_code
#   File "D:\PROJECTS\ANN_project_deployment\venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
#   File "D:\PROJECTS\ANN_project_deployment\venv\Lib\site-packages\tensorboard\main.py", line 27, in <module>
#     from tensorboard import default
#   File "D:\PROJECTS\ANN_project_deployment\venv\Lib\site-packages\tensorboard\default.py", line 30, in <module>
#     import pkg_resources
# ModuleNotFoundError: No module named 'pkg_resources'


# solve the problem broken pkg_resources import in tensorboard by

# pip uninstall setuptools -y
# pip uninstall setuptools -y
# pip cache purge
# pip install --no-cache-dir setuptools==80.9.0
# python -c "import pkg_resources; print('OK')"

In [34]:
import pkg_resources
print("OK")

OK


C:\Users\User\AppData\Local\Temp\ipykernel_5360\2647389871.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [35]:
%load_ext tensorboard

In [36]:
# The tensorboard extension is already loaded. To reload it, use:
%reload_ext tensorboard

In [37]:
%tensorboard --logdir logs/fit20260617-022007

Reusing TensorBoard on port 6006 (pid 22936), started 13:40:44 ago. (Use '!kill 22936' to kill it.)